In [0]:
df = spark.read.json("/Volumes/ai_logs_catalog/logs_schema/raw_logs_vol/logs/")
display(df)

In [0]:
from pyspark.sql.types import StructType, StringType, IntegerType
from pyspark.sql.functions import *

# Creating schema to infer.
log_schema = StructType() \
    .add("timestamp", StringType()) \
    .add("service_name", StringType()) \
    .add("log_level", StringType()) \
    .add("message", StringType()) \
    .add("error_code", StringType()) \
    .add("response_time", IntegerType())

# Reading Data using AutoLoader.
bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(log_schema)
    .load("/Volumes/ai_logs_catalog/logs_schema/raw_logs_vol/logs")
)

enr_bronze_df = bronze_df \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", input_file_name())

# Writing data in append mode (delta).
(enr_bronze_df.writeStream
 .format("delta")
 .outputMode("append")
 .trigger(availableNow=True)
 .option("checkpointLocation", "/Volumes/ai_logs_catalog/logs_schema/checkpoints/bronze_logs")
 .toTable("ai_logs_catalog.logs_schema.bronze_logs")
)

